# T33 / E15 — Đối chứng ngoài trên ViWikiFC, split gốc

Chấm phương pháp trên **đúng split mà ViWikiFC phát hành**, để con số đặt cạnh được các mốc đã
công bố ở mục 6 `docs/EXPERIMENTS.md`.

| Việc | Thời gian |
|---|---|
| Chuẩn bị dữ liệu, kiểm môi trường | ~2 phút, CPU |
| Trích đặc trưng 20.919 mẫu | **~2 giờ 30** |
| Soi shard, lấy kết quả về | ~1 phút, CPU |

**Không có ô dò kiểu số** — mô hình đọc là Qwen2.5-7B, lớp tràn số đã chốt ở T07 là lớp 27 và ghi
vào mục 3 `CLAUDE.md`. **Không có ô đo chi phí** — `measure_throughput` ở T31 đã cho phép chiếu
cho ViWikiFC từ trung bình theo mức độ dài.

## Ba điều phải ghi kèm mọi con số của phiên này

1. **Tập test dùng lại 100 % ngữ cảnh của train.** Đo ở T14, xem `results/leakage_report.md`.
   Điểm trên đó **không nói gì** về dữ liệu chưa từng thấy. Các mốc đã công bố cũng chịu đúng
   phần rò rỉ ấy nên phép so vẫn hợp lệ — nhưng cả hai bên đều không đo được khái quát hóa.
2. **Chỉ 67 % nhãn NEI thật sự là ngoại lai**, kappa 0,505, đo ở T13. Lớp `extrinsic` của bộ này
   nhiễu hơn hẳn ViHallu.
3. **5,9 % mẫu chỉ có một đoạn**, nơi chunk-aware thoái hóa thành lookback gộp. Đo lại ngày
   10/09 trên đủ 20.919 mẫu.

## Sau khi chạy xong

Tải cả thư mục `ket_qua_t33` về, rồi báo lại. Chấm ở **máy cá nhân**, 0 giây GPU — quy tắc chốt ở
T23: mọi phép so phải chấm trên cùng một máy vì điểm dev lệch tới 0,0075 giữa hai môi trường.


In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode:
        print(result.stderr.strip())
        raise SystemExit(f"lệnh hỏng: {' '.join(str(a) for a in args)}")
    return result.stdout


if REPO_DIR.exists():
    run("git", "fetch", "--all", cwd=REPO_DIR)
    run("git", "reset", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR))

os.chdir(REPO_DIR)
run("git", "log", "-1", "--format=%h %s")

$ git clone --depth 1 https://github.com/wsunicorn/vihallulens.git /kaggle/working/vihallulens

$ git log -1 --format=%h %s
ebc9ae3 T33: dựng công cụ E15 trên split gốc ViWikiFC, và sửa một con số của T15 (#97)


'ebc9ae3 T33: dựng công cụ E15 trên split gốc ViWikiFC, và sửa một con số của T15 (#97)\n'

In [2]:
# Ô 2 — hai cấu hình. Không phải sửa gì ở ô này.
#
# Hai file dung chung mot extraction_hash vi hash chi tinh tren dataset, chunking va extractor —
# khong tinh features. Nen moc lookback KHONG ton them giay GPU nao, no doc lai dung shard.
CAU_HINH = {
    "chunk": "configs/e15_chunk_viwikifc.yaml",
    "moc": "configs/e15_baseline_lookback_viwikifc.yaml",
}
BO_DU_LIEU = "viwikifc"
CAC_TAP = ("train", "dev", "test")

print("=" * 78)
for khoa, duong_dan in CAU_HINH.items():
    print(f"  {khoa:<6} {duong_dan}")
print("=" * 78)
print("  Split GOC cua bo, khong chia lai. train 16.738 / dev 2.090 / test 2.091.")

  chunk  configs/e15_chunk_viwikifc.yaml
  moc    configs/e15_baseline_lookback_viwikifc.yaml
  Split GOC cua bo, khong chia lai. train 16.738 / dev 2.090 / test 2.091.


In [3]:
# Ô 3 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit.
!pip install -q --no-deps -e .
!pip install -q bitsandbytes

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 31.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.
vihallulens 0.1.0 requires rank-bm25, which is not installed.


In [4]:
# Ô 4 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
import importlib.util
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config
from vihallulens.data.paths import find_raw_dir

problems = []

packages = ("torch", "transformers", "bitsandbytes", "pandas", "accelerate")
absent = [name for name in packages if importlib.util.find_spec(name) is None]
print(f"  goi phai co san   : {f'THIEU {absent}' if absent else f'du ca {list(packages)}'}")
if absent:
    problems.append(f"thieu goi {absent}")

try:
    raw = find_raw_dir()
    files = sorted(p.name for p in Path(raw).glob("viwikifc*"))
    print(f"  du lieu tho       : {raw}")
    print(f"  file viwikifc tho : {files or 'KHONG CO'}")
    if len(files) < 3:
        problems.append(f"can ba file viwikifc_train/dev/test, chi thay {files}")
except Exception as error:
    print(f"  du lieu tho       : KHONG TIM THAY ({error})")
    problems.append("chua mount dataset du lieu tho")

# Hai hash phai trung NGAY BAY GIO. Khong co o nao ghi de exclude_layers trong phien nay — lop 27
# da chot tu T07 — nen lech o day la loi cau hinh, va biet truoc thi re hon biet sau 2,5 gio GPU.
hashes = {}
for khoa, duong_dan in CAU_HINH.items():
    cfg = load_config(duong_dan)
    hashes[khoa] = extraction_hash(cfg)
    print(f"  {khoa:<6} exclude {str(cfg.extractor.exclude_layers):<6} "
          f"nhom {cfg.features.groups}  hash {hashes[khoa]}")
if len(set(hashes.values())) != 1:
    problems.append(f"HAI HASH KHAC NHAU {hashes} — moc lookback se doi trich lai tu dau")

chain = [
    ("o 5", "data/interim/viwikifc_{train,dev,test}.parquet", "normalize_data + split_data"),
    ("o 6", "data/processed/viwikifc_{split}_<hash>.jsonl", "extract_features, ~2 gio 30"),
    ("o 7", "so mau co lop tran so", "inspect_shard"),
    ("o 8", "ket_qua_t33/ — CAT KET QUA DAT TIEN DI TRUOC", "shutil.copy"),
]
print("  chuoi tu tao:")
for cell, target, maker in chain:
    print(f"    {cell:<6} {target:<48} <- {maker}")

if problems:
    raise SystemExit("TIEN KIEM HONG: " + "; ".join(problems))
print("\nTien kiem dat.")

  goi phai co san   : du ca ['torch', 'transformers', 'bitsandbytes', 'pandas', 'accelerate']
  du lieu tho       : /kaggle/input/datasets/unicorn1209/vihallulens
  file viwikifc tho : ['viwikifc_dev.csv', 'viwikifc_test.csv', 'viwikifc_train.csv']
  chunk  exclude [27]   nhom ['basic', 'chunk_aware', 'stability']  hash 3c1a9a052afa
  moc    exclude [27]   nhom ['basic']  hash 3c1a9a052afa
  chuoi tu tao:
    o 5    data/interim/viwikifc_{train,dev,test}.parquet   <- normalize_data + split_data
    o 6    data/processed/viwikifc_{split}_<hash>.jsonl     <- extract_features, ~2 gio 30
    o 7    so mau co lop tran so                            <- inspect_shard
    o 8    ket_qua_t33/ — CAT KET QUA DAT TIEN DI TRUOC     <- shutil.copy

Tien kiem dat.


In [5]:
# Ô 5 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, CPU.
#
# split_data giu NGUYEN split goc cho viwikifc (KEEP_ORIGINAL trong scripts/split_data.py) — no
# chi chep qua va bao cao ro ri, khong chia lai theo ngu canh nhu voi vihallu.
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset viwikifc
!python scripts/split_data.py --only viwikifc
!python -m pytest tests/test_attention_hook.py tests/test_viwikifc.py -q
!python -m pytest tests/test_drop_nonfinite.py tests/test_chunking.py -q


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : ebc9ae3 T33: dựng công cụ E15 trên split gốc ViWikiFC, và sửa một con số của T15 (#97)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.0.0
  bitsandbytes     : 0.50.2
  accelerate       : 1.13.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA VIWIKIFC
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  số dòng               : 20

## Trích đặc trưng

20.919 mẫu, một lượt duy nhất phục vụ **cả hai** cấu hình vì chúng trùng `extraction_hash`.

Ngữ cảnh bộ này ngắn — trung bình 154 từ, 4,73 đoạn — nên không mẫu nào chạm trần 4.096 token.
Chi phí vì thế nằm gần trọn ở mức `0–512`, khoảng 424 ms mỗi mẫu theo phép chiếu của T31.

In [6]:
# Ô 6 — TRÍCH ĐẶC TRƯNG. Khoảng 2 giờ 30. Chạy lại được, có lưu tiến độ.
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTHONUNBUFFERED"] = "1"

HONG = []
for tap in CAC_TAP:
    print("=" * 78)
    print(f"  TRICH — {BO_DU_LIEU}/{tap}")
    print("=" * 78)
    ma = os.system(f"python scripts/extract_features.py --config {CAU_HINH['chunk']} --split {tap}")
    if ma:
        # KHONG raise. Trich co luu tien do nen shard do dang van dang mang ve: phien sau doc
        # tiep tu cho do thay vi chay lai tu dau. Raise o day thi o 7 va o 8 khong chay, va phan
        # do dang bi bo lai tren may Kaggle. Bai hoc T30 va T31.
        HONG.append(f"{tap} (ma loi {ma})")
        print(f"  !! {tap} hong, ma loi {ma} — ghi lai va di tiep")

if HONG:
    print()
    print("  " + "!" * 74)
    print(f"  CO {len(HONG)} TAP HONG: {', '.join(HONG)}")
    print("  Van chay tiep o 7 va o 8 de mang ve phan da trich duoc.")
    print("  " + "!" * 74)

  TRICH — viwikifc/train

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e15_chunk_viwikifc.yaml  (hash 3c1a9a052afa)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : viwikifc/train, 16,738 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 16,737/16,738  → ghi thêm gold_rank cho E06
  đã có sẵn             : 0 mẫu trong viwikifc_train_3c1a9a052afa.jsonl
  còn phải chạy         : 16,738 mẫu


      16738/16738  395 ms/mẫu  còn ~0 phút  lỗi 0

--------------------------------------------------------------------------------
  đã ghi thêm           : 16,738 mẫu, tổng 16,738
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/16,738
  có lớp tràn số        : 0/16,738
  thời gian             : 110.1 phút, 395 ms/mẫu
  file                  : data/processed/viwikifc_train_3c1a9a052afa.jsonl
  TRICH — viwikifc/dev

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e15_chunk_viwikifc.yaml  (hash 3c1a9a052afa)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : viwikifc/dev, 2,090 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 2,090/2,090  → ghi thêm gold_rank cho E06
  đã có sẵn             

       2090/2090  397 ms/mẫu  còn ~0 phút  lỗi 0

--------------------------------------------------------------------------------
  đã ghi thêm           : 2,090 mẫu, tổng 2,090
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/2,090
  có lớp tràn số        : 0/2,090
  thời gian             : 13.8 phút, 397 ms/mẫu
  file                  : data/processed/viwikifc_dev_3c1a9a052afa.jsonl
  TRICH — viwikifc/test

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e15_chunk_viwikifc.yaml  (hash 3c1a9a052afa)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : viwikifc/test, 2,091 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 2,091/2,091  → ghi thêm gold_rank cho E06
  đã có sẵn             : 0 mẫ

       2091/2091  400 ms/mẫu  còn ~0 phút  lỗi 0

--------------------------------------------------------------------------------
  đã ghi thêm           : 2,091 mẫu, tổng 2,091
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/2,091
  có lớp tràn số        : 0/2,091
  thời gian             : 14.0 phút, 400 ms/mẫu
  file                  : data/processed/viwikifc_test_3c1a9a052afa.jsonl


In [7]:
# Ô 7 — soi shard. Vài giây, CPU. KHÔNG dừng notebook.
#
# Khong raise du shard co nan: mot shard hong la thu CAN dem ve nhat de chan doan. Bai hoc T30.
proc = subprocess.run(["python", "scripts/inspect_shard.py", "--config", CAU_HINH["chunk"]],
                      capture_output=True, text=True)
print(proc.stdout[-6000:])
if proc.returncode:
    print(proc.stderr[-2000:])
print()
print("  DU SHARD CO NAN HAY KHONG, VAN CHAY O 8 DE MANG VE.")


SOI SHARD — e15_chunk_viwikifc
  mô hình đọc       : Qwen/Qwen2.5-7B-Instruct
  đang bỏ lớp       : [27]
  hash trích        : 3c1a9a052afa

  TRAIN  16,738 mẫu   lưới 27 × 28
    mẫu có lớp tràn số : 0  (0.00 %)

  DEV  2,090 mẫu   lưới 27 × 28
    mẫu có lớp tràn số : 0  (0.00 %)

  TEST  2,091 mẫu   lưới 27 × 28
    mẫu có lớp tràn số : 0  (0.00 %)

------------------------------------------------------------------------------------
  Không lớp nào tràn số. Shard sạch.


  DU SHARD CO NAN HAY KHONG, VAN CHAY O 8 DE MANG VE.


## Chấm điểm — KHÔNG chạy ở đây

Quy tắc chốt ở T23: mọi phép so sánh phải chấm trên **cùng một máy**, vì điểm dev lệch tới 0,0075
giữa Kaggle và máy cá nhân do bộ giải tối ưu hội tụ khác nhau.

In [8]:
# Ô 8 — LẤY KẾT QUẢ VỀ. Vài giây. Ô cuối, và là ô quan trọng nhất của phiên.
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config

out = Path("/kaggle/working/ket_qua_t33")
out.mkdir(exist_ok=True)

run_hash = extraction_hash(load_config(CAU_HINH["chunk"]))
for tap in CAC_TAP:
    src = Path(f"data/processed/{BO_DU_LIEU}_{tap}_{run_hash}.jsonl")
    if src.exists():
        shutil.copy(src, out / src.name)
    else:
        print(f"  !! thieu {src}")
for duong_dan in CAU_HINH.values():
    shutil.copy(duong_dan, out / Path(duong_dan).name)

# CHEP CA HAI SO KET QUA. O 10 cua T31 chi chep runs.jsonl, ma measure_throughput ghi vao
# feasibility.jsonl, nen sau luot do chi phi nam lai tren may Kaggle va phai tai bu mot lan nua.
for so in ("results/runs.jsonl", "results/feasibility.jsonl"):
    p = Path(so)
    if p.exists():
        shutil.copy(p, out / f"{p.stem}_t33{p.suffix}")
for bao_cao in Path("results").glob("leakage_report*"):
    shutil.copy(bao_cao, out / bao_cao.name)

for f in sorted(out.iterdir()):
    print(f"  {f.name:<46} {f.stat().st_size / 1e6:>8.1f} MB")

print("""
Tai het thu muc ket_qua_t33 ve may, dat vao:
  viwikifc_*.jsonl     ->  data/processed/
  *.yaml               ->  configs/   (doi chieu, khong bat buoc ghi de)
  runs_t33.jsonl       ->  giu lai
  feasibility_t33.jsonl->  giu lai
  leakage_report*.md   ->  results/

Roi bao lai de cham diem o may ca nhan, 0 giay GPU. Cham moc lookback TRUOC.
""")

  e15_baseline_lookback_viwikifc.yaml                 0.0 MB
  e15_chunk_viwikifc.yaml                             0.0 MB
  feasibility_t33.jsonl                               0.0 MB
  leakage_report.md                                   0.0 MB
  leakage_report_viwikifc.md                          0.0 MB
  runs_t33.jsonl                                      0.4 MB
  viwikifc_dev_3c1a9a052afa.jsonl                   113.2 MB
  viwikifc_test_3c1a9a052afa.jsonl                  113.1 MB
  viwikifc_train_3c1a9a052afa.jsonl                 903.4 MB

Tai het thu muc ket_qua_t33 ve may, dat vao:
  viwikifc_*.jsonl     ->  data/processed/
  *.yaml               ->  configs/   (doi chieu, khong bat buoc ghi de)
  runs_t33.jsonl       ->  giu lai
  feasibility_t33.jsonl->  giu lai
  leakage_report*.md   ->  results/

Roi bao lai de cham diem o may ca nhan, 0 giay GPU. Cham moc lookback TRUOC.

